# Handwritten Digits Generator With a GAN

This jupyter notebook follow the previous one 'GAN to generate data accroding to a sinus distribution'.

We're going to use a 'Vanilla' GAN (not effective, especially for the image) to generate images of handwritten digits. For that, we’ll train the models using the MNIST dataset of handwritten digits, which is included in the torchvision package. 

Since this example uses images in the training set, the models need to be more complex, with a larger number of parameters. This makes the training process slower. To reduce it, you can use a GPU to train the model if you have one available. This jupyter notebook show how to manage cpu and gpu.


## Import the necessary libraries
- torchvision include the MNIST dataset
- torchvision.transforms is used to perform image conversions
- Torchinfo provides information complementary to what is provided by print(your_model) and help to visualize the model
- matplotlib.pyplot for graph

In [ ]:
import torch
from torch import nn

import math

import torchvision
import torchvision.transforms as transforms 

from torchinfo import summary

import matplotlib.pyplot as plt
from IPython.display import clear_output

### Check for GPU device
You can ensure your code will run on cpu and gpu by creating a device object that points either to the CPU or, if one is available, to the GPU. For recent MAC you can test for MPS.

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

#Additional Info when using cuda
if device == 'cuda':
    print('Number of GPU(s):', torch.cuda.device_count())
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

Later, we’ll use this device to set where tensors and models should be created, using the GPU if available.

## Parameters

In [ ]:
# Root directory for dataset
dataroot = "data"

# Batch size to be utilized
batch_size = 32

# learning rate 
lr = 0.0001 

# number of epochs
num_epochs = 200 

# image size
im_size = 28

# latent vector size
latent_size = 100

# Display step 
disp_setp = 10

## Data

### load and prepare the data
The MNIST dataset consists of 28 × 28 pixel grayscale images of handwritten digits from 0 to 9. To use them with PyTorch, you’ll need to perform some conversions. For that, you define transform, a function to be used when loading the data.

In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)

The function has two parts:

* transforms.ToTensor() converts the data to a PyTorch tensor.

* transforms.Normalize() converts the range of the tensor coefficients.

The original coefficients given by transforms.ToTensor() range from 0 to 1, and since the image backgrounds are black, most of the coefficients are equal to 0 when they’re represented using this range.

transforms.Normalize() changes the range of the coefficients to -1 to 1 by subtracting 0.5 from the original coefficients and dividing the result by 0.5. With this transformation, the number of elements equal to 0 in the input samples is dramatically reduced, which helps in training the models. Plus, this is a common data processing with deep learning models.

The arguments of transforms.Normalize() are two tuples, (M₁, ..., Mₙ) and (S₁, ..., Sₙ), with n representing the number of channels of the images. Grayscale images such as those in MNIST dataset have only one channel (n=1), so the tuples have only one value. Then, for each channel i of the image, transforms.Normalize() subtracts Mᵢ from the coefficients and divides the result by Sᵢ.

Now you can load the training data using torchvision.datasets.MNIST and perform the conversions using transform:

In [ ]:
train_set = torchvision.datasets.MNIST(
    root=dataroot, train=True, download=True, transform=transform
)
# train=True select only the training data
# download=True ensures that the first time you run the above code
# the MNIST dataset will be downloaded and stored in the current directory, as indicated by the argument root, or to another directory
# tranform = pre-process the data

print('Dataset size = ', len(train_set))

Number of sample by class

In [ ]:
nums = [0]*10
for i in range(len(train_set)):
  nums[(int(train_set.targets[i]))] += 1
print(nums)

Note: balanced classes are generally preferable. To manage unbalanced classes, there are suitable loss functions

Reduce the number of data to speed up the training

In [ ]:
tr_split_len = 10000
tr_set = torch.utils.data.random_split(train_set, [tr_split_len, len(train_set)-tr_split_len])[0]

nums = [0]*10
for i in range(tr_split_len):
  nums[(int(tr_set.dataset.targets[tr_set.indices[i]]))] += 1
print(nums)

### Create the data loader

In [ ]:
train_loader = torch.utils.data.DataLoader(
    tr_set, batch_size=batch_size, shuffle=True, drop_last=True
)
# drop_last=True to drop the last incomplete batch when the number of examples are not exactly divided by the batch size
# otherwise we have to manage the size of the batch in the rest of the code (this can be done without to much difficulties)

### Show some training data 

In [ ]:
real_samples, mnist_labels = next(iter(train_loader))
for i in range(16):
    ax = plt.subplot(4, 4, i + 1)
    plt.imshow(real_samples[i].reshape(28, 28), cmap="gray_r") # cmap=gray_r to reverse the color map 
    plt.xticks([])
    plt.yticks([])

As you can see, there are digits with different handwriting styles. As the GAN learns the distribution of the data, it’ll also generate digits with different handwriting styles.

## Implementation

### Implementing the Discriminator
The discriminator is an MLP neural network that receives a 28 × 28 pixel image and provides the probability of the image belonging to the real training data. Note that the images must be flatten/vectorized so that the neural network receives vectors with 784 coefficients.

The original shape of the input x in 'forward' is 32 × 1 × 28 × 28, where 32 is the batch size. After the conversion, the shape of x becomes 32 × 784, with each line representing the coefficients of an image of the training set.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(im_size*im_size, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = x.view(x.size(0), im_size*im_size) # vectorization of the input image: from [batch_sizex1x28x28] to [batch_sizex784]
        output = self.model(x)
        return output

### Implementing the Generator
Since the generator is going to generate more complex data, it’s necessary to increase the dimensions of the input from the latent space. In this case, the generator is going to be fed a 100-dimensional input and will provide an output with 784 coefficients, which will be organized in a 1 x 28 × 28 tensor representing an image.

Note that we could directly construct the dataset with the flattened image, reducing the number of operation to be done during training.

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_size, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, im_size*im_size),
            nn.Tanh(), # the output coefficients should be in the interval from -1 to 1
        )

    def forward(self, x):
        output = self.model(x)
        output = output.view(x.size(0), 1, im_size,im_size) # reshape to original image size [1x28x28]
        return output

### Instantiate the Discriminator and Generator objects

To run the discriminator and generator models using the GPU, we have to instantiate it and send it to the GPU with .to(). To use a GPU, when there’s one available, you can send the model to the device object created earlier

In [ ]:
discriminator = Discriminator().to(device=device)
generator = Generator().to(device=device)

### Visualization of the models

As in the previous jupyter notebook, we can use print to display the models. Here we use another tool called summary.

In [ ]:
# it seems that when using summary the model is sent to GPU
if device=='cuda':
    print(summary(discriminator,(batch_size,784))) 
else:
    print(discriminator)

In [ ]:
if device=='cuda':
    print(summary(generator,(batch_size,latent_size)))
else:
    print(generator)

### Loss and optimizers

Define the loss

In [ ]:
loss_function = nn.BCELoss() #  binary cross-entropy function used to train the model

Define the optimizers

In [ ]:
optimizer_discriminator = torch.optim.Adam(discriminator.parameters(), lr=lr)
optimizer_generator = torch.optim.Adam(generator.parameters(), lr=lr)

### Custom weights initialization

In [ ]:

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)
    elif classname.find('Linear') != -1:
        nn.init.xavier_uniform_(m.weight.data)

generator.apply(weights_init)
discriminator.apply(weights_init)

### Training loop

It is very similar to the one we used in the previous example. Note that we need to send the training data to device to use the GPU, if available.

Some of the tensors don’t need to be sent to the GPU explicitly with device. This is the case with generated_samples, which will already be sent to an available GPU since latent_space_samples and generator were sent to the GPU previously.

Note: if you want to run the training loop several times from scratch do not forgot to run the weigths initialization

In [ ]:
# Set models to train mode
generator.train()
discriminator.train()

# define the graph
fig, ax = plt.subplots(4, 4, constrained_layout=True)

# training loop
for epoch in range(num_epochs):
    for n, (real_samples, mnist_labels) in enumerate(train_loader):
        #---------- Data for training the discriminator ----------#
        real_samples = real_samples.to(device)
        
        # since we don't use the class of the data (see cGAN) we need to redefine the labels and set them to 1
        real_samples_labels = torch.ones((batch_size, 1)).to(device)

        # define the latent space
        latent_space_samples = torch.randn((batch_size, latent_size)).to(device)
        
        # Disabling gradient calculation for inference in the block (temporarily sets all of the requires_grad flags to false)
        # reduce memory consumption for computations (the gradients are not used during discriminator training)
        with torch.no_grad():
            generated_samples = generator(latent_space_samples) # fake data

        # create the labels with value 0 for the generated samples
        generated_samples_labels = torch.zeros((batch_size, 1)).to(device)

        #---------- Training the discriminator  ----------#
        discriminator.zero_grad()

        # calculate the output of the discriminator
        output_real = discriminator(real_samples)
        output_fake = discriminator(generated_samples)

        # calculate the loss function using the output from the model (predicted labels) and the true labels
        loss_real = loss_function(output_real, real_samples_labels)
        loss_fake = loss_function(output_fake, generated_samples_labels)

        loss_discriminator=(loss_real+loss_fake)/2

        # calculate the gradients to update the weights
        loss_discriminator.backward()
        
        #loss_discriminator.backward()
        optimizer_discriminator.step()       

        #---------- Training the generator  ----------#
        # define the latent space
        latent_space_samples = torch.randn((batch_size, latent_size)).to(device)

        # clear the gradients
        generator.zero_grad()

        # generate new data
        generated_samples = generator(latent_space_samples)

        # calculate the output of the discriminator using the generated data
        output_discriminator_generated = discriminator(generated_samples)

        # calculate the loss function using the output of the classification system and the labels, which are all equal to 1
        # to fool the discriminator: Non-saturating loss
        loss_generator = loss_function(output_discriminator_generated, real_samples_labels)

        # calculate the gradients
        loss_generator.backward()

        # update the generator weights
        optimizer_generator.step()
        
    # Show sometime the discriminator and generator loss
    if epoch % disp_setp == 0:
        # Turn generator to eval mode
        generator.eval()
            
        # generate new data
        latent_space_samples = torch.randn(batch_size, latent_size).to(device)
        with torch.no_grad():
            generated_samples = generator(latent_space_samples)

        # display some generated image
        generated_samples = generated_samples.cpu()
        for i in range(16):
            ax = plt.subplot(4, 4, i + 1)
            plt.imshow(generated_samples[i].reshape(28, 28), cmap="gray_r")
            plt.xticks([])
            plt.yticks([])
        fig.suptitle('Epoch {}, Generator loss {:.4f}, Discriminator loss {:.4f}'.format((epoch), loss_discriminator, loss_generator))
        display(fig)
        clear_output(wait=True)
            
        # Turn back generator to train mode
        generator.train()



## Display new samples generated by the GAN

In [ ]:
# Turn Networks to eval mode
generator.eval()

# generate new data
latent_space_samples = torch.randn(batch_size, 100).to(device)
with torch.no_grad():
    generated_samples = generator(latent_space_samples)

To plot generated_samples, we need to move the data back to the CPU in case it’s running on the GPU. For that, you can simply call .cpu(). As we did previously, we also need to call .detach() before using Matplotlib to plot the data (if torch.no_grad is not used)

In [ ]:
generated_samples = generated_samples.cpu()
for i in range(16):
    ax = plt.subplot(4, 4, i + 1)
    plt.imshow(generated_samples[i].reshape(28, 28), cmap="gray_r")
    plt.xticks([])
    plt.yticks([])